# Coral Detection Demo

This notebook compares the baseline YOLOv8n model with the final white-balance + CLAHE pipeline on selected held-out coral images.

It does not train any model. It only loads saved weights and runs predictions for the live demo.

In [ ]:
# Connect Google Drive and check that a T4 GPU is available

from google.colab import drive
drive.mount("/content/drive")

import torch
from pathlib import Path

assert torch.cuda.is_available(), (
    "No GPU detected. Go to Runtime > Change runtime type > T4 GPU, "
    "then reconnect and run this cell again."
)

PROJECT_ROOT = Path("/content/drive/MyDrive/CSBP441_Coral_Project")

assert PROJECT_ROOT.is_dir(), f"Project folder not found: {PROJECT_ROOT}"

print("GPU:", torch.cuda.get_device_name(0))
print("Project folder found:", PROJECT_ROOT)

## 1. Load the required software

We use the same Ultralytics version as the final experiments so the saved YOLOv8n weights load consistently.

In [ ]:
# Install the exact YOLO version used for the final experiments

%pip install -q ultralytics==8.4.207

import ultralytics
import cv2
import matplotlib.pyplot as plt

assert ultralytics.__version__ == "8.4.207"

print("Ultralytics:", ultralytics.__version__)
print("OpenCV:", cv2.__version__)

## 2. Check saved demo assets

We use saved model weights and previously selected held-out examples. No training or full test-set scan is performed during the live demo.

In [ ]:
# Check the final saved models, test images, and prepared demo examples

import json

BASELINE_WEIGHTS = (
    PROJECT_ROOT
    / "baseline"
    / "runs"
    / "yolov8n_baseline_35class_seed42"
    / "weights"
    / "best.pt"
)

COMBINED_WEIGHTS = (
    PROJECT_ROOT
    / "combined"
    / "runs"
    / "yolov8n_combined_wb_clahe_35class_seed42"
    / "weights"
    / "best.pt"
)

RAW_TEST_IMAGES = (
    PROJECT_ROOT
    / "data_balanced"
    / "Coral-Detection-35class-balanced"
    / "test"
    / "images"
)

RAW_TEST_LABELS = (
    PROJECT_ROOT
    / "data_balanced"
    / "Coral-Detection-35class-balanced"
    / "test"
    / "labels"
)

DEMO_LOG = PROJECT_ROOT / "combined" / "demo" / "demo_examples.json"

required_files = {
    "Baseline weights": BASELINE_WEIGHTS,
    "Combined weights": COMBINED_WEIGHTS,
    "Test images": RAW_TEST_IMAGES,
    "Test labels": RAW_TEST_LABELS,
    "Saved demo examples": DEMO_LOG,
}

for name, path in required_files.items():
    print(f"{'FOUND' if path.exists() else 'MISSING'} — {name}: {path}")

assert all(path.exists() for path in required_files.values()), (
    "One or more demo assets are missing. Do not continue yet."
)

demo_examples = json.loads(DEMO_LOG.read_text())

print("\nSaved demo examples:")
for example in demo_examples:
    print(
        f"- {example['kind']}: {example['image']} | "
        f"GT: {example['ground_truth']} | "
        f"Baseline: {example['baseline_pred']} | "
        f"Proposed: {example['combined_pred']}"
    )

## 3. Copy only demo assets locally

Copying two saved model weights and three fixed examples to local Colab storage makes the live demo faster and avoids slow Drive access during prediction.

In [ ]:
# Copy final model weights and three fixed demo examples to local Colab storage

import shutil

LOCAL_DEMO = Path("/content/coral_demo")
LOCAL_WEIGHTS = LOCAL_DEMO / "weights"
LOCAL_IMAGES = LOCAL_DEMO / "images"
LOCAL_LABELS = LOCAL_DEMO / "labels"

for folder in [LOCAL_WEIGHTS, LOCAL_IMAGES, LOCAL_LABELS]:
    folder.mkdir(parents=True, exist_ok=True)

# Use two improvement examples and one limitation example.
selected_examples = [
    demo_examples[1],  # Rare Anacropa: baseline wrong, proposed correct
    demo_examples[0],  # Favia: baseline wrong, proposed correct
    demo_examples[3],  # Goniastrea: baseline correct, proposed wrong
]

# Copy final weights locally.
LOCAL_BASELINE_WEIGHTS = LOCAL_WEIGHTS / "baseline_best.pt"
LOCAL_COMBINED_WEIGHTS = LOCAL_WEIGHTS / "combined_best.pt"

shutil.copy2(BASELINE_WEIGHTS, LOCAL_BASELINE_WEIGHTS)
shutil.copy2(COMBINED_WEIGHTS, LOCAL_COMBINED_WEIGHTS)

# Copy only the selected images and their labels locally.
for example in selected_examples:
    image_name = example["image"]
    label_name = f"{Path(image_name).stem}.txt"

    shutil.copy2(RAW_TEST_IMAGES / image_name, LOCAL_IMAGES / image_name)
    shutil.copy2(RAW_TEST_LABELS / label_name, LOCAL_LABELS / label_name)

print("Local baseline weights:", LOCAL_BASELINE_WEIGHTS)
print("Local combined weights:", LOCAL_COMBINED_WEIGHTS)

print("\nSelected live-demo examples:")
for example in selected_examples:
    print(f"- {example['kind']}: {example['image']} | GT: {example['ground_truth']}")

## 4. Load the saved models

The models are loaded once and warmed up before the audience sees the predictions. The proposed pipeline applies gray-world white balance, then CLAHE.

In [ ]:
# Load the two saved models and define the final proposed preprocessing pipeline

from ultralytics import YOLO
import cv2
import yaml
import numpy as np

DATA_YAML = (
    PROJECT_ROOT
    / "data_balanced"
    / "Coral-Detection-35class-balanced"
    / "data.yaml"
)

class_names = yaml.safe_load(DATA_YAML.read_text())["names"]

baseline_model = YOLO(str(LOCAL_BASELINE_WEIGHTS))
combined_model = YOLO(str(LOCAL_COMBINED_WEIGHTS))

PREDICT_ARGS = {
    "imgsz": 640,
    "conf": 0.25,
    "iou": 0.7,
    "max_det": 300,
    "device": 0,
    "verbose": False,
}

clahe = cv2.createCLAHE(clipLimit=1.0, tileGridSize=(4, 4))

def gray_world_white_balance(image_bgr):
    """Reduce global underwater colour cast."""
    pixels = image_bgr.astype(np.float32)
    channel_means = pixels.mean(axis=(0, 1))
    gains = np.ones(3, dtype=np.float32)
    np.divide(channel_means.mean(), channel_means, out=gains,
              where=channel_means > 0)
    return np.clip(np.rint(pixels * gains), 0, 255).astype(np.uint8)

def apply_clahe_lab(image_bgr):
    """Apply CLAHE only to the LAB lightness channel."""
    L, A, B = cv2.split(cv2.cvtColor(image_bgr, cv2.COLOR_BGR2LAB))
    enhanced = cv2.merge((clahe.apply(L), A, B))
    return cv2.cvtColor(enhanced, cv2.COLOR_LAB2BGR)

def proposed_preprocess(image_bgr):
    return apply_clahe_lab(gray_world_white_balance(image_bgr))

# Warm up both models once before the live demonstration.
warmup_path = LOCAL_IMAGES / selected_examples[0]["image"]
warmup_image = cv2.imread(str(warmup_path))

baseline_model.predict(warmup_image, **PREDICT_ARGS)
combined_model.predict(proposed_preprocess(warmup_image), **PREDICT_ARGS)

torch.cuda.synchronize()

print("Baseline and proposed models loaded and warmed up.")
print("Classes:", len(class_names))

## 5. Display one comparison

This function runs one selected image through both models and shows the baseline output beside the proposed WB + CLAHE output. Blue means the predicted class matches the ground truth; red means it does not.

In [ ]:
# Create a reusable function for one side-by-side live comparison

from time import perf_counter
from matplotlib.patches import Rectangle
import matplotlib.pyplot as plt

def get_ground_truth_id(image_name):
    label_path = LOCAL_LABELS / f"{Path(image_name).stem}.txt"
    lines = [line for line in label_path.read_text().splitlines() if line.strip()]
    return int(float(lines[0].split()[0]))

def get_top_detection(result):
    if result.boxes is None or len(result.boxes) == 0:
        return None, None, None

    top_index = int(result.boxes.conf.argmax().item())
    class_id = int(result.boxes.cls[top_index].item())
    confidence = float(result.boxes.conf[top_index].item())
    box = result.boxes.xyxy[top_index].cpu().numpy()

    return class_id, confidence, box

def draw_prediction(ax, image_bgr, result, ground_truth_id, title):
    ax.imshow(cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB))
    ax.axis("off")
    ax.set_title(title, fontsize=12, fontweight="bold")

    class_id, confidence, box = get_top_detection(result)

    if class_id is None:
        ax.text(
            10, 30, "No detection",
            fontsize=12, color="black",
            bbox=dict(facecolor="yellow", alpha=0.9, edgecolor="none")
        )
        return "No detection"

    correct = class_id == ground_truth_id
    color = "deepskyblue" if correct else "red"

    x1, y1, x2, y2 = box
    ax.add_patch(Rectangle(
        (x1, y1), x2 - x1, y2 - y1,
        fill=False, edgecolor=color, linewidth=3
    ))

    prediction_text = f"{class_names[class_id]} ({confidence:.2f})"
    ax.text(
        x1 + 5, max(y1 + 25, 25), prediction_text,
        fontsize=11, fontweight="bold", color="black",
        bbox=dict(facecolor=color, alpha=0.9, edgecolor="none")
    )

    return prediction_text

def show_demo_example(example_index):
    """Show one fixed example: baseline versus WB + CLAHE."""
    example = selected_examples[example_index]
    image_name = example["image"]
    image_path = LOCAL_IMAGES / image_name
    image_bgr = cv2.imread(str(image_path))

    assert image_bgr is not None, f"Could not load {image_name}"

    ground_truth_id = get_ground_truth_id(image_name)
    ground_truth_name = class_names[ground_truth_id]

    # Baseline timing
    torch.cuda.synchronize()
    start = perf_counter()
    baseline_result = baseline_model.predict(image_bgr, **PREDICT_ARGS)[0]
    torch.cuda.synchronize()
    baseline_ms = (perf_counter() - start) * 1000

    # Proposed-pipeline timing: preprocessing plus YOLO inference
    torch.cuda.synchronize()
    start = perf_counter()
    processed_bgr = proposed_preprocess(image_bgr)
    proposed_result = combined_model.predict(processed_bgr, **PREDICT_ARGS)[0]
    torch.cuda.synchronize()
    proposed_ms = (perf_counter() - start) * 1000

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    baseline_text = draw_prediction(
        axes[0], image_bgr, baseline_result, ground_truth_id,
        f"Baseline YOLOv8n\n{baseline_ms:.1f} ms"
    )

    axes[1].imshow(cv2.cvtColor(processed_bgr, cv2.COLOR_BGR2RGB))
    axes[1].axis("off")
    axes[1].set_title("WB + CLAHE image", fontsize=12, fontweight="bold")

    proposed_text = draw_prediction(
        axes[2], processed_bgr, proposed_result, ground_truth_id,
        f"Proposed WB + CLAHE + YOLOv8n\n{proposed_ms:.1f} ms"
    )

    fig.suptitle(f"Ground truth: {ground_truth_name}", fontsize=16, fontweight="bold")
    fig.text(
        0.5, 0.02,
        "Blue = correct class   |   Red = wrong class",
        ha="center", fontsize=11
    )
    plt.tight_layout(rect=[0, 0.05, 1, 0.92])
    plt.show()

    print("Image:", image_name)
    print("Ground truth:", ground_truth_name)
    print("Baseline:", baseline_text)
    print("Proposed:", proposed_text)

## 6. Live demo: improvement on a rare class

The first example shows the rare coral class Anacropa. The baseline predicts the wrong class, while the proposed pipeline predicts Anacropa correctly.

In [ ]:
show_demo_example(0)

## 7. Second improvement example

The second example shows another image where the proposed pipeline corrects the baseline classification.

In [ ]:
show_demo_example(1)

## 8. Limitation example

The final example shows that preprocessing does not always help. It is included to demonstrate the system’s limitation honestly.

In [ ]:
show_demo_example(2)

## 9. Overall test-set results

The following table summarizes the four-pipeline comparison on the same 169 held-out test images. These are the official measurements used in the report.

In [ ]:
# Display a clean official four-pipeline comparison table

import pandas as pd

ABLATION_TABLE = (
    PROJECT_ROOT
    / "combined"
    / "results"
    / "ablation_four_pipelines.csv"
)

results_table = pd.read_csv(ABLATION_TABLE)

# Rename the saved CSV index column for presentation.
results_table = results_table.rename(
    columns={results_table.columns[0]: "Pipeline"}
)

demo_results = results_table[
    ["Pipeline", "precision", "recall", "mAP50", "mAP50_95"]
].copy()

demo_results.columns = [
    "Pipeline", "Precision", "Recall", "mAP@50", "mAP@50-95"
]

display(
    demo_results.style.format({
        "Precision": "{:.3f}",
        "Recall": "{:.3f}",
        "mAP@50": "{:.3f}",
        "mAP@50-95": "{:.3f}",
    })
)

print("Key takeaway: CLAHE alone achieved the highest mAP@50 (0.480).")
print("The combined pipeline improved over baseline (0.402 → 0.438) but exceeded the latency target.")

## 10. Backup plan

If the Colab runtime fails, use the saved comparison images below. These were generated in advance from the same baseline and proposed models.

In [ ]:
# Show the saved backup demo figures without running either model

from IPython.display import display, Image

BACKUP_DEMO_FOLDER = PROJECT_ROOT / "combined" / "demo"
backup_images = sorted(BACKUP_DEMO_FOLDER.glob("demo_*.png"))

assert backup_images, f"No backup demo images found in {BACKUP_DEMO_FOLDER}"

print("Saved backup demo figures:")
for image_path in backup_images:
    print("-", image_path.name)
    display(Image(filename=str(image_path)))